In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from diffusers import StableDiffusionPipeline
import torch

In [ ]:
#handpicked_HS
df_HP_HS = pd.read_csv("handpicked_HS.csv")
df_HP_HS.head()

#this will add a new column to the dataframe with the image links
#it will be in the format of images/IHS_####.jpg so it will use a for loop to increment the number for each row.
#the {i:04d} will add leading zeros to the number so that it is always 4 digits long.
df_HP_HS['image'] = [f"images/IHS_{i:04d}.jpg" for i in range(1, len(df_HP_HS)+1)]

#save the dataframe to a new csv file
df_HP_HS.to_csv("handpicked_HS_Img.csv", index=False)

In [ ]:
#DE_Gen_HS
df_DE_HS = pd.read_csv("DE_Gen_HS.csv")

#this will add a new column to the dataframe with the image links
#it will be in the format of images/IHS_####.jpg so it will use a for loop to increment the number for each row.
#the {i:04d} will add leading zeros to the number so that it is always 4 digits long.
df_DE_HS['image'] = [f"images_DE/DE_HS_{i:04d}.jpg" for i in range(1, len(df_DE_HS)+1)]

#save the dataframe to a new csv file
df_DE_HS.to_csv("DE_Gen_HS_Img.csv", index=False)


In [ ]:
#read csv file into dataframe
df_HS_Gen = pd.read_csv("SD_Gen_HS.csv")  
#store the text column as a list of prompts

df_HS_Gen['image'] = [f"Images_SB/SD_HS_{i:04d}.jpg" for i in range(1, len(df_HS_Gen)+1)]
#save the dataframe to a new csv file
df_HS_Gen.to_csv("SD_Gen_HS_Img.csv", index=False)
prompts = df_HS_Gen["Text"].dropna().tolist()[:1299]
#print the length of the prompts list
print(f"Loaded {len(prompts)} prompts.")

In [ ]:
#create a pipeline for stable diffusion
pipe = StableDiffusionPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V6.0_B1_noVAE",
    torch_dtype=torch.float16
).to("cuda")

In [ ]:
#creates a target directory if it doesn't exist
#sets the output directory for generated images
output_dir = "Images_SB"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
#this will generate images based directly on the text prompts
#and save them in the output directory

#loops through prompt list with a progress bar
for i, prompt in tqdm(enumerate(prompts), total=len(prompts)):
    #try block to catch any errors during image generation
    try:
        #generate image from prompt from the pipeline
        #num_inference_steps is how many steps the model takes to generate the image higher is usually better but slower
        #guidance_scale controls how closely the image matches the prompt higher values mean closer match but can reduce diversity
        image = pipe(prompt,
                     negative_prompt= "nude, nudity, naked, cleavage,Breasts, nipples, suggestive, explicit, deformed, low quality",
                     num_inference_steps= 50,
                     guidance_scale= 7.5).images[0] 
        #save the image with a filename based on the index
        image.save(os.path.join(output_dir, f"SD_HS_{i+1:04}.png"))
    #except block to handle exceptions
    except Exception as e:
        print(f"Error at index {i}: {e}")

In [ ]:
#This will generate images based on the text prompts with a "hopeful" theme added onto it
#This is to see if adding a theme will improve the quality of the images
hope_suffix = ", hopeful, uplifting, warm colors, soft glow, sense of peace"
hope_prompts = [p + hope_suffix for p in prompts]

for i, prompt in tqdm(enumerate(hope_prompts), total=len(hope_prompts)):
    try:
        image = pipe(prompt,
                     negative_prompt= "nude, nudity, naked, cleavage,Breasts, nipples, suggestive, explicit, deformed, low quality",
                     num_inference_steps= 50,
                     guidance_scale= 7.5).images[0]
        image.save(os.path.join(output_dir, f"SD_HS_{i+1:04}.jpg"))
    except Exception as e:
        print(f"Error at index {i}: {e}")